# Notebook 11 — Hybrid Retrieval API Integration

This notebook validates the **Phase 11 code already integrated in the FastAPI project**, not a separate hybrid implementation.

The API defaults to Dense E5. Use `POST /search` with `"mode": "hybrid"` for BM25 Full + Dense E5 + RRF. `/ask` remains extractive and dense-only.

The following **single executable cell** runs lightweight unit and API tests, style checks, and the entire test suite using mocked search/classification models. It does **not** download model weights, rebuild indexes, alter benchmark artifacts or rerun Notebook 08. It may apply source-code formatting to Phase 11 Python files.

Run from a local Jupyter environment with the repository (and its `pyproject.toml`) available; install `python -m pip install -e ".[dev]"` beforehand.


In [ ]:
# ONE-CELL PHASE 11 VALIDATION — no GPU, no model inference, no artifact writes
from pathlib import Path
import subprocess
import sys

root = Path.cwd()
if not (root / "pyproject.toml").exists():
    root = root.parent
assert (root / "pyproject.toml").exists(), "Open Jupyter from the repository root or notebooks/."

format_cmd = [
    sys.executable, "-m", "ruff", "format",
    "src/greek_nlp/api/main.py", "src/greek_nlp/api/schemas.py",
    "src/greek_nlp/rag/hybrid.py", "tests/test_hybrid.py",
]
print("RUN:", " ".join(format_cmd), flush=True)
subprocess.run(format_cmd, cwd=root, check=True)

checks = [
    [sys.executable, "-m", "pytest", "tests/test_hybrid.py", "-q"],
    [sys.executable, "-m", "pytest", "-q"],
    [sys.executable, "-m", "ruff", "check", "."],
    [sys.executable, "-m", "ruff", "format", "--check", "."],
]
for cmd in checks:
    print("\nRUN:", " ".join(cmd), flush=True)
    result = subprocess.run(cmd, cwd=root, check=False)
    if result.returncode:
        raise RuntimeError(f"Validation failed ({result.returncode}): {cmd}")
print("\nPHASE 11 CHECKS PASSED. Live HF model/index smoke remains a separate optional step.")


## Optional live smoke test

After setting `HF_MODEL_REPO` and `HF_INDEX_REPO` and installing `[ml]`, run the existing API locally and compare two `POST /search` requests (one without `mode`, one with `"mode": "hybrid"`). See [`docs/phase_11_hybrid_integration.md`](../docs/phase_11_hybrid_integration.md). The first hybrid request builds BM25 from the deployed `docs.json`. **Do not treat the historical Notebook 08 scores as newly reproduced by this integration test.**
